In [ ]:
import polars as pl

data = pl.read_parquet("../data/1m/1m.parquet").lazy()

In [ ]:
data.collect_schema()

In [ ]:
data1 = (
    data.filter(pl.col("symbol") == "BTCUSDT")
    .select("open_time", pl.col("close").log().alias("BTC"))
    .join(
        data.filter(pl.col("symbol") == "ETHUSDT").select(
            "open_time", pl.col("close").log().alias("ETH")
        ),
        on="open_time",
    )
)

In [ ]:
data2 = data1.filter((pl.col("open_time")>=pl.datetime(2021,1,1)) & (pl.col("open_time")<pl.datetime(2021,1,4)))

In [ ]:
import statsmodels.api as sm

X = data2.select("ETH").collect().to_numpy()
y = data2.select("BTC").collect().to_numpy()
model = sm.OLS(y, sm.add_constant(X))

results = model.fit()
print(results.summary())


In [ ]:
pl.from_numpy(results.resid).with_columns(pl.arange(0, results.resid.shape[0]).alias("index")).plot.line(y="column_0",x="index")

In [ ]:
data2.collect_schema()

In [ ]:
data2.with_columns(
    pl.rolling_cov("BTC", "ETH", window_size=60).alias("rolling_cov"),
    pl.col("ETH").rolling_var(window_size=60).alias("rolling_var_eth"),
).with_columns(
    (pl.col("rolling_cov") / pl.col("rolling_var_eth")).alias("rolling_beta"),
).with_columns(
    (
        pl.col("BTC").rolling_mean(window_size=60)
        - pl.col("rolling_beta") * pl.col("ETH").rolling_mean(window_size=60)
    ).alias("rolling_alpha"),
).with_columns(
    (
        pl.col("BTC")
        - (pl.col("rolling_alpha") + pl.col("rolling_beta") * pl.col("ETH"))
    ).alias("rolling_residual")
).drop_nulls().collect().plot.line(y="rolling_residual", x="open_time")

In [ ]:
resids = data2.with_columns(
    pl.rolling_cov("BTC", "ETH", window_size=60).alias("rolling_cov"),
    pl.col("ETH").rolling_var(window_size=60).alias("rolling_var_eth"),
).with_columns(
    (pl.col("rolling_cov") / pl.col("rolling_var_eth")).alias("rolling_beta"),
).with_columns(
    (
        pl.col("BTC").rolling_mean(window_size=60)
        - pl.col("rolling_beta") * pl.col("ETH").rolling_mean(window_size=60)
    ).alias("rolling_alpha"),
).with_columns(
    (
        pl.col("BTC")
        - (pl.col("rolling_alpha") + pl.col("rolling_beta") * pl.col("ETH"))
    ).alias("rolling_residual")
).drop_nulls().select("rolling_residual").collect().to_series()

from statsmodels.tsa.stattools import adfuller

adfuller(resids)

In [ ]:
data2.with_columns(
    ((pl.col("BTC") * pl.col("ETH")).ewm_mean(half_life=15)-pl.col("BTC").ewm_mean(half_life=15)*pl.col("ETH").ewm_mean(half_life=15)).alias("ewm_cov"),
    (pl.col("ETH").ewm_var(half_life=15)).alias("ewm_var_eth"),
).with_columns(
    (pl.col("ewm_cov") / pl.col("ewm_var_eth")).alias("ewm_beta"),
).with_columns(
    (
        pl.col("BTC").ewm_mean(half_life=15)
        - pl.col("ewm_beta") * pl.col("ETH").ewm_mean(half_life=15)
    ).alias("ewm_alpha"),
).with_columns(
    (
        pl.col("BTC")
        - (pl.col("ewm_alpha") + pl.col("ewm_beta") * pl.col("ETH"))
    ).alias("ewm_residual")
).drop_nulls().collect().plot.line(y="ewm_residual", x="open_time")


In [ ]:
data2.with_columns(
    ((pl.col("BTC") * pl.col("ETH")).ewm_mean(half_life=15)-pl.col("BTC").ewm_mean(half_life=15)*pl.col("ETH").ewm_mean(half_life=15)).alias("ewm_cov"),
    (pl.col("ETH").ewm_var(half_life=15)).alias("ewm_var_eth"),
).with_columns(
    (pl.col("ewm_cov") / pl.col("ewm_var_eth")).alias("ewm_beta"),
).with_columns(
    (
        pl.col("BTC").ewm_mean(half_life=15)
        - pl.col("ewm_beta") * pl.col("ETH").ewm_mean(half_life=15)
    ).alias("ewm_alpha"),
).with_columns(
    (
        pl.col("BTC")
        - (pl.col("ewm_alpha") + pl.col("ewm_beta") * pl.col("ETH"))
    ).alias("ewm_residual")
).drop_nulls().drop_nans().with_columns(pl.col("ewm_residual").cum_sum().alias("ewm_residual_cumsum")).collect().plot.line(y="ewm_residual_cumsum", x="open_time")


In [ ]:
resids = data2.with_columns(
    ((pl.col("BTC") * pl.col("ETH")).ewm_mean(half_life=15)-pl.col("BTC").ewm_mean(half_life=15)*pl.col("ETH").ewm_mean(half_life=15)).alias("ewm_cov"),
    (pl.col("ETH").ewm_var(half_life=15)).alias("ewm_var_eth"),
).with_columns(
    (pl.col("ewm_cov") / pl.col("ewm_var_eth")).alias("ewm_beta"),
).with_columns(
    (
        pl.col("BTC").ewm_mean(half_life=15)
        - pl.col("ewm_beta") * pl.col("ETH").ewm_mean(half_life=15)
    ).alias("ewm_alpha"),
).with_columns(
    (
        pl.col("BTC")
        - (pl.col("ewm_alpha") + pl.col("ewm_beta") * pl.col("ETH"))
    ).alias("ewm_residual")
).drop_nulls().drop_nans().select("ewm_residual").collect().to_numpy()


adfuller(resids)

In [ ]:
resids = data2.with_columns(
    pl.rolling_cov("BTC", "ETH", window_size=60).alias("rolling_cov"),
    pl.col("ETH").rolling_var(window_size=60).alias("rolling_var_eth"),
).with_columns(
    (pl.col("rolling_cov") / pl.col("rolling_var_eth")).alias("rolling_beta"),
).with_columns(
    (
        pl.col("BTC").rolling_mean(window_size=60)
        - pl.col("rolling_beta") * pl.col("ETH").rolling_mean(window_size=60)
    ).alias("rolling_alpha"),
).with_columns(
    (
        pl.col("BTC")
        - (pl.col("rolling_alpha") + pl.col("rolling_beta") * pl.col("ETH"))
    ).alias("rolling_residual")
).drop_nulls().drop_nans().select(pl.col("open_time"), pl.col("rolling_residual").cum_sum().alias("rolling_residual_cumsum")).select("rolling_residual_cumsum").collect().to_numpy()

adfuller(resids)

In [ ]:
adfuller(data2.with_columns(
    pl.rolling_cov("BTC", "ETH", window_size=60).alias("rolling_cov"),
    pl.col("ETH").rolling_var(window_size=60).alias("rolling_var_eth"),
).with_columns(
    (pl.col("rolling_cov") / pl.col("rolling_var_eth")).alias("rolling_beta"),
).with_columns(
    (
        pl.col("BTC").rolling_mean(window_size=60)
        - pl.col("rolling_beta") * pl.col("ETH").rolling_mean(window_size=60)
    ).alias("rolling_alpha"),
).with_columns(
    (
        pl.col("BTC")
        - (pl.col("rolling_alpha") + pl.col("rolling_beta") * pl.col("ETH"))
    ).alias("rolling_residual")
).with_columns(
    (
        pl.col("rolling_residual")
        - pl.col("rolling_residual").rolling_mean(window_size=60)
    ).alias("rolling_residual_demeaned")
).drop_nulls().drop_nans().select(
    pl.col("open_time"),
    pl.col("rolling_residual_demeaned").cum_sum().alias("rolling_residual_cumsum"),
).select("rolling_residual_cumsum").collect().to_numpy())